CRIME HOTSPOT MAPPING-DASHBOARD

---



DATASET LOADING

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


INSTALL THE LIBRARIES

In [54]:
!pip install -q streamlit duckdb pandas numpy matplotlib seaborn plotly scikit-learn

In [55]:
import glob
import os

files = glob.glob(
    '/content/drive/MyDrive/**/chicago_scaled.csv',
    recursive=True
)

if not files:
    files = glob.glob(
        '/content/drive/MyDrive/**/Chicago_Crimes_2001_to_Present.csv',
        recursive=True
    )

if not files:
    raise FileNotFoundError("Crime dataset not found in Google Drive.")

DATA_PATH = files[0]

print("Dataset found!")
print("Location:", DATA_PATH)
print("Size:", round(os.path.getsize(DATA_PATH) / (1024**3), 2), "GB")

Dataset found!
Location: /content/drive/MyDrive/chicago_scaled.csv
Size: 3.12 GB


In [56]:
import duckdb

con = duckdb.connect('/content/crime_dashboard.duckdb')

safe_path = DATA_PATH.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW crimes AS
    SELECT *
    FROM read_csv_auto('{safe_path}', header=true, sample_size=100000)
""")

print("DuckDB connected to the crime dataset successfully!")

DuckDB connected to the crime dataset successfully!


In [57]:
columns = con.execute("DESCRIBE crimes").df()

print("Number of columns:", len(columns))
display(columns)

Number of columns: 35


,column_name,column_type,null,key,default,extra
0,ID,BIGINT,YES,None,None,None
1,Case Number,VARCHAR,YES,None,None,None
2,Date,TIMESTAMP,YES,None,None,None
3,Block,VARCHAR,YES,None,None,None
4,IUCR,VARCHAR,YES,None,None,None
5,Primary Type,VARCHAR,YES,None,None,None
6,Description,VARCHAR,YES,None,None,None
7,Location Description,VARCHAR,YES,None,None,None
8,Arrest,BOOLEAN,YES,None,None,None
9,Domestic,BOOLEAN,YES,None,None,None


DASHBOARD CODE



In [108]:
%%writefile app.py

import streamlit as st
import duckdb
import os
import pandas as pd
import pydeck as pdk
import plotly.express as px
import plotly.graph_objects as go


# ==========================================
# 1. PAGE CONFIGURATION
# ==========================================

st.set_page_config(
    page_title="Night Watch | Crime Dashboard",
    page_icon="🌃",
    layout="wide"
)

DATA_PATH = os.environ.get("CRIME_CSV")


# ==========================================
# 2. DATABASE CONNECTION
# ==========================================

@st.cache_resource
def get_connection(path):

    db = duckdb.connect()

    safe_path = path.replace("'", "''")

    db.execute(f"""
        CREATE VIEW crimes AS
        SELECT *
        FROM read_csv_auto(
            '{safe_path}',
            header=true,
            sample_size=100000
        )
    """)

    return db


# ==========================================
# 3. COMPACT DASHBOARD DESIGN
# ==========================================

st.markdown("""
<style>

.block-container {
    padding-top: 1rem;
    padding-bottom: 1rem;
}

h1 {
    font-size: 1.8rem !important;
    margin-bottom: 0.1rem !important;
}

h2 {
    font-size: 1.3rem !important;
    margin-top: 0.2rem !important;
    margin-bottom: 0.3rem !important;
}

h3 {
    font-size: 1rem !important;
    margin-bottom: 0.2rem !important;
}

div[data-testid="stVerticalBlock"] {
    gap: 0.65rem;
}

.stApp {
    background-color: #0B1020;
    color: #F1F5F9;
}

p, label {
    color: #CBD5E1;
}

[data-testid="stMetric"] {
    background-color: #151D33;
    padding: 12px;
    border-radius: 10px;
    border: 1px solid #26334F;
}

[data-testid="stMetricValue"] {
    color: #60A5FA;
    font-size: 1.5rem;
}

[data-testid="stMetricLabel"] {
    color: #CBD5E1;
    font-size: 0.85rem;
}

hr {
    margin-top: 0.5rem;
    margin-bottom: 0.5rem;
}

</style>
""", unsafe_allow_html=True)


# ==========================================
# 4. DASHBOARD HEADER
# ==========================================

st.markdown("# 🌃 NIGHT WATCH")
st.markdown("### Crime Intelligence & Hotspot Mapping")
st.caption("Visualizing Crime Patterns for Safer Cities")


# ==========================================
# 5. DATASET VALIDATION
# ==========================================

if not DATA_PATH or not os.path.exists(DATA_PATH):

    st.error("Dataset path not configured. Check CRIME_CSV.")
    st.stop()


try:

    con = get_connection(DATA_PATH)


    # ======================================
    # 6. FILTER OPTIONS
    # ======================================

    @st.cache_data
    def get_filter_options():

        years = con.execute("""
            SELECT DISTINCT Year
            FROM crimes
            WHERE Year IS NOT NULL
            ORDER BY Year
        """).fetchdf()["Year"].tolist()

        months = con.execute("""
            SELECT DISTINCT Month
            FROM crimes
            WHERE Month IS NOT NULL
            ORDER BY Month
        """).fetchdf()["Month"].tolist()

        categories = con.execute("""
            SELECT DISTINCT "Primary Type"
            FROM crimes
            WHERE "Primary Type" IS NOT NULL
            ORDER BY "Primary Type"
        """).fetchdf()["Primary Type"].tolist()

        districts = con.execute("""
            SELECT DISTINCT District
            FROM crimes
            WHERE District IS NOT NULL
            ORDER BY District
        """).fetchdf()["District"].tolist()

        periods = con.execute("""
            SELECT DISTINCT TimePeriod
            FROM crimes
            WHERE TimePeriod IS NOT NULL
            ORDER BY TimePeriod
        """).fetchdf()["TimePeriod"].tolist()

        return years, months, categories, districts, periods


    years, months, categories, districts, periods = get_filter_options()

    if not years:
        st.error("No year values found in the dataset.")
        st.stop()


    # ======================================
    # 7. GLOBAL FILTERS - FIVE FILTERS
    # ======================================

    st.markdown("## 🔎 Global Filters")

    year_range = st.slider(
        "Year Range",
        min_value=int(min(years)),
        max_value=int(max(years)),
        value=(int(min(years)), int(max(years)))
    )

    f1, f2 = st.columns(2)

    with f1:

        selected_months = st.multiselect(
            "Month",
            months,
            format_func=lambda x: str(int(x)),
            placeholder="All months"
        )

    with f2:

        selected_categories = st.multiselect(
            "Crime Category",
            categories,
            placeholder="All categories"
        )

    f3, f4 = st.columns(2)

    with f3:

        selected_districts = st.multiselect(
            "District",
            districts,
            format_func=lambda x: str(int(x)),
            placeholder="All districts"
        )

    with f4:

        selected_periods = st.multiselect(
            "Time Period",
            periods,
            placeholder="All time periods"
        )


    # ======================================
    # 8. BUILD FILTER CONDITIONS
    # ======================================

    conditions = ['"Year" BETWEEN ? AND ?']

    params = [year_range[0], year_range[1]]

    if selected_months:

        conditions.append(
            '"Month" IN (' +
            ','.join(['?'] * len(selected_months)) + ')'
        )

        params.extend(selected_months)

    if selected_categories:

        conditions.append(
            '"Primary Type" IN (' +
            ','.join(['?'] * len(selected_categories)) + ')'
        )

        params.extend(selected_categories)

    if selected_districts:

        conditions.append(
            '"District" IN (' +
            ','.join(['?'] * len(selected_districts)) + ')'
        )

        params.extend(selected_districts)

    if selected_periods:

        conditions.append(
            '"TimePeriod" IN (' +
            ','.join(['?'] * len(selected_periods)) + ')'
        )

        params.extend(selected_periods)

    where_clause = " AND ".join(conditions)


    # ======================================
    # 9. KPI CARDS
    # ======================================

    st.divider()

    st.markdown("## 📊 Crime Statistics Overview")

    with st.spinner("Calculating crime statistics..."):

        kpi_data = con.execute(
            f"""
            SELECT
                COUNT(*) AS total,

                AVG(
                    CASE WHEN "Arrest" = TRUE
                    THEN 1.0 ELSE 0.0 END
                ) * 100 AS arrest_rate,

                SUM(
                    CASE WHEN "Domestic" = TRUE
                    THEN 1 ELSE 0 END
                ) AS domestic_count

            FROM crimes
            WHERE {where_clause}
            """,
            params
        ).fetchone()

        total = kpi_data[0] or 0
        arrest_rate = kpi_data[1] or 0
        domestic_count = kpi_data[2] or 0

        common = con.execute(
            f"""
            SELECT "Primary Type", COUNT(*) AS total
            FROM crimes
            WHERE {where_clause}
            GROUP BY "Primary Type"
            ORDER BY total DESC
            LIMIT 1
            """,
            params
        ).fetchone()

        district = con.execute(
            f"""
            SELECT District, COUNT(*) AS total
            FROM crimes
            WHERE {where_clause}
              AND District IS NOT NULL
            GROUP BY District
            ORDER BY total DESC
            LIMIT 1
            """,
            params
        ).fetchone()

        peak = con.execute(
            f"""
            SELECT Hour, COUNT(*) AS total
            FROM crimes
            WHERE {where_clause}
              AND Hour IS NOT NULL
            GROUP BY Hour
            ORDER BY total DESC
            LIMIT 1
            """,
            params
        ).fetchone()


    k1, k2, k3 = st.columns(3)

    k1.metric("Total Crime Incidents", f"{total:,}")
    k2.metric("Arrest Rate", f"{arrest_rate:.2f}%")
    k3.metric("Domestic Incidents", f"{domestic_count:,}")

    k4, k5, k6 = st.columns(3)

    k4.metric(
        "Most Common Crime",
        common[0] if common else "N/A"
    )

    district_value = (
        str(int(district[0]))
        if district and float(district[0]).is_integer()
        else str(district[0]) if district else "N/A"
    )

    k5.metric("Highest Crime District", district_value)

    peak_value = f"{int(peak[0]):02d}:00" if peak else "N/A"

    k6.metric("Peak Crime Hour", peak_value)


    # ======================================
    # 10. PREPARE GRAPH DATA
    # ======================================

    st.divider()

    st.markdown("## 📈 Crime Pattern Visualizations")

    with st.spinner("Preparing interactive graphs..."):

        # Year-wise data

        year_data = con.execute(
            f"""
            SELECT Year, COUNT(*) AS Incidents
            FROM crimes
            WHERE {where_clause}
            GROUP BY Year
            ORDER BY Year
            """,
            params
        ).fetchdf()


        # Crime category data

        crime_data = con.execute(
            f"""
            SELECT
                "Primary Type",
                COUNT(*) AS Incidents
            FROM crimes
            WHERE {where_clause}
            GROUP BY "Primary Type"
            ORDER BY Incidents DESC
            LIMIT 10
            """,
            params
        ).fetchdf()


        # Time period data

        period_data = con.execute(
            f"""
            SELECT
                TimePeriod,
                COUNT(*) AS Incidents
            FROM crimes
            WHERE {where_clause}
            GROUP BY TimePeriod
            ORDER BY Incidents DESC
            """,
            params
        ).fetchdf()


        # District data: Arrest vs Non-Arrest

        district_data = con.execute(
            f"""
            SELECT
                CAST(District AS VARCHAR) AS District,

                SUM(
                    CASE WHEN "Arrest" = TRUE
                    THEN 1 ELSE 0 END
                ) AS Arrest,

                SUM(
                    CASE WHEN "Arrest" = FALSE
                    THEN 1 ELSE 0 END
                ) AS Non_Arrest

            FROM crimes

            WHERE {where_clause}
              AND District IS NOT NULL

            GROUP BY District
            ORDER BY District
            """,
            params
        ).fetchdf()


    # ======================================
    # 11. TWO-COLUMN GRAPH LAYOUT
    # ======================================

    graph1, graph2 = st.columns(2)


    # --------------------------------------
    # GRAPH 1: LINE CHART
    # --------------------------------------

    with graph1:

        st.markdown("### 1. Year-wise Crime Trend")

        if not year_data.empty:

            st.line_chart(
                year_data.set_index("Year"),
                y="Incidents",
                height=280
            )

        else:

            st.info("No data available.")


    # --------------------------------------
    # GRAPH 2: HORIZONTAL LOLLIPOP CHART
    # --------------------------------------

    with graph2:

        st.markdown("### 2. Top 10 Crime Categories")

        if not crime_data.empty:

            lollipop_fig = go.Figure()

            # Draw horizontal lines

            for _, row in crime_data.iterrows():

                lollipop_fig.add_trace(
                    go.Scatter(
                        x=[0, row["Incidents"]],
                        y=[row["Primary Type"], row["Primary Type"]],
                        mode="lines",
                        line=dict(
                            color="#64748B",
                            width=2
                        ),
                        showlegend=False,
                        hoverinfo="skip"
                    )
                )

            # Add circular markers

            lollipop_fig.add_trace(
                go.Scatter(
                    x=crime_data["Incidents"],
                    y=crime_data["Primary Type"],
                    mode="markers",
                    marker=dict(
                        size=11,
                        color="#38BDF8"
                    ),
                    name="Incidents",
                    hovertemplate=(
                        "Crime: %{y}<br>"
                        "Incidents: %{x:,}<extra></extra>"
                    )
                )
            )

            lollipop_fig.update_layout(
                height=350,
                margin=dict(l=10, r=10, t=20, b=20),
                xaxis_title="Number of Incidents",
                yaxis_title="",
                showlegend=False,
                plot_bgcolor="#0B1020",
                paper_bgcolor="#0B1020",
                font=dict(color="#CBD5E1")
            )

            st.plotly_chart(
                lollipop_fig,
                use_container_width=True
            )

        else:

            st.info("No crime category data available.")


    # --------------------------------------
    # GRAPH 3: DONUT CHART
    # --------------------------------------

    graph3, graph4 = st.columns(2)

    with graph3:

        st.markdown("### 3. Crime by Time Period")

        if not period_data.empty:

            donut_fig = px.pie(
                period_data,
                names="TimePeriod",
                values="Incidents",
                hole=0.55
            )

            donut_fig.update_traces(
                textposition="inside",
                textinfo="percent+label",
                hovertemplate=(
                    "%{label}<br>"
                    "Incidents: %{value:,}<br>"
                    "Percentage: %{percent}<extra></extra>"
                )
            )

            donut_fig.update_layout(
                height=350,
                margin=dict(l=10, r=10, t=20, b=20),
                plot_bgcolor="#0B1020",
                paper_bgcolor="#0B1020",
                font=dict(color="#CBD5E1"),
                legend_title_text="Time Period"
            )

            st.plotly_chart(
                donut_fig,
                use_container_width=True
            )

        else:

            st.info("No time period data available.")


    # --------------------------------------
    # GRAPH 4: STACKED BAR CHART
    # --------------------------------------

    with graph4:

        st.markdown("### 4. Crime by District")

        if not district_data.empty:

            district_long = district_data.melt(
                id_vars=["District"],
                value_vars=["Arrest", "Non_Arrest"],
                var_name="Arrest Status",
                value_name="Incidents"
            )

            district_long["Arrest Status"] = (
                district_long["Arrest Status"].replace({
                    "Non_Arrest": "Non-Arrest"
                })
            )

            stacked_fig = px.bar(
                district_long,
                x="District",
                y="Incidents",
                color="Arrest Status",
                barmode="stack",
                labels={
                    "District": "District",
                    "Incidents": "Number of Incidents"
                }
            )

            stacked_fig.update_layout(
                height=350,
                margin=dict(l=10, r=10, t=20, b=20),
                plot_bgcolor="#0B1020",
                paper_bgcolor="#0B1020",
                font=dict(color="#CBD5E1"),
                legend_title_text="Arrest Status"
            )

            st.plotly_chart(
                stacked_fig,
                use_container_width=True
            )

        else:

            st.info("No district data available.")


    # ======================================
    # 12. YEAR-TO-YEAR CRIME COMPARISON
    # ======================================

    st.divider()

    st.markdown("## 📅 Year-to-Year Crime Comparison")

    with st.spinner("Comparing yearly crime incidents..."):

        yearly_comparison = con.execute(
            f"""
            SELECT
                Year,
                COUNT(*) AS Incidents
            FROM crimes
            WHERE {where_clause}
            GROUP BY Year
            ORDER BY Year
            """,
            params
        ).fetchdf()


    if len(yearly_comparison) > 0:

        yearly_comparison["Year"] = (
            yearly_comparison["Year"].astype(int)
        )

        yearly_comparison["Change"] = (
            yearly_comparison["Incidents"].diff()
        )

        yearly_comparison["Percentage Change"] = (
            yearly_comparison["Incidents"].pct_change() * 100
        )


        # ----------------------------------
        # GRAPH 5: AREA CHART
        # ----------------------------------

        st.markdown("### Annual Crime Incidents")

        area_fig = px.area(
            yearly_comparison,
            x="Year",
            y="Incidents",
            labels={
                "Year": "Year",
                "Incidents": "Number of Incidents"
            }
        )

        area_fig.update_layout(
            height=300,
            margin=dict(l=10, r=10, t=20, b=20),
            plot_bgcolor="#0B1020",
            paper_bgcolor="#0B1020",
            font=dict(color="#CBD5E1")
        )

        st.plotly_chart(
            area_fig,
            use_container_width=True
        )


        # ----------------------------------
        # YEAR-TO-YEAR COMPARISON TABLE
        # ----------------------------------

        st.markdown("### Annual Increase or Decrease")

        comparison_table = yearly_comparison.copy()

        comparison_table["Change"] = (
            comparison_table["Change"].apply(
                lambda x: (
                    "First year"
                    if pd.isna(x)
                    else f"{x:+,.0f}"
                )
            )
        )

        comparison_table["Percentage Change"] = (
            comparison_table["Percentage Change"].apply(
                lambda x: (
                    "N/A"
                    if pd.isna(x)
                    else f"{x:+.2f}%"
                )
            )
        )

        st.dataframe(
            comparison_table,
            use_container_width=True,
            hide_index=True
        )


        if len(yearly_comparison) >= 2:

            latest = yearly_comparison.iloc[-1]
            previous = yearly_comparison.iloc[-2]

            change = latest["Percentage Change"]

            if change > 0:

                st.warning(
                    f"Crime incidents increased by {change:.2f}% "
                    f"from {int(previous['Year'])} to "
                    f"{int(latest['Year'])}."
                )

            elif change < 0:

                st.success(
                    f"Crime incidents decreased by "
                    f"{abs(change):.2f}% from "
                    f"{int(previous['Year'])} to "
                    f"{int(latest['Year'])}."
                )

            else:

                st.info("Crime incidents remained unchanged.")

    else:

        st.info("No yearly comparison data available.")


    # ======================================
    # 13. CRIME DENSITY HEATMAP + HOTSPOT MAP
    # ======================================

    st.divider()

    st.markdown("## 🗺️ Crime Density & Hotspot Mapping")

    st.caption(
        "View the concentration of crime incidents and their individual "
        "locations side by side. Both maps follow the global filters above."
    )


    with st.spinner("Loading geographical crime data..."):

        # Individual crime locations

        map_data = con.execute(
            f"""
            SELECT
                Latitude,
                Longitude
            FROM crimes
            WHERE {where_clause}
              AND Latitude BETWEEN 41.0 AND 43.0
              AND Longitude BETWEEN -88.0 AND -87.0
              AND Latitude IS NOT NULL
              AND Longitude IS NOT NULL
            LIMIT 5000
            """,
            params
        ).fetchdf()


        # Crime density data

        heatmap_data = con.execute(
            f"""
            SELECT
                ROUND(Latitude, 3) AS Latitude,
                ROUND(Longitude, 3) AS Longitude,
                COUNT(*) AS Crime_Count
            FROM crimes
            WHERE {where_clause}
              AND Latitude BETWEEN 41.0 AND 43.0
              AND Longitude BETWEEN -88.0 AND -87.0
              AND Latitude IS NOT NULL
              AND Longitude IS NOT NULL
            GROUP BY
                ROUND(Latitude, 3),
                ROUND(Longitude, 3)
            """,
            params
        ).fetchdf()


    heat_col, hotspot_col = st.columns(2, gap="medium")


    # --------------------------------------
    # LEFT: CRIME DENSITY HEATMAP
    # --------------------------------------

    with heat_col:

        st.markdown("### 🔥 Crime Density Heatmap")

        st.caption(
            "Brighter regions indicate a higher concentration of incidents."
        )

        if not heatmap_data.empty:

            heat_layer = pdk.Layer(
                "HeatmapLayer",
                data=heatmap_data,
                get_position="[Longitude, Latitude]",
                get_weight="Crime_Count",
                radius_pixels=40,
                intensity=1,
                threshold=0.05,
                pickable=True
            )

            heat_view = pdk.ViewState(
                latitude=41.8781,
                longitude=-87.6298,
                zoom=10,
                pitch=0
            )

            heat_deck = pdk.Deck(
                layers=[heat_layer],
                initial_view_state=heat_view,
                tooltip={
                    "text": "Crime incidents: {Crime_Count}"
                }
            )

            st.pydeck_chart(
                heat_deck,
                use_container_width=True,
                height=420
            )

        else:

            st.info(
                "No geographical density data for the selected filters."
            )


    # --------------------------------------
    # RIGHT: CRIME HOTSPOT MAP
    # --------------------------------------

    with hotspot_col:

        st.markdown("### 📍 Crime Hotspot Map")

        st.caption(
            "Each point represents a recorded crime incident."
        )

        if not map_data.empty:

            st.map(
                map_data,
                latitude="Latitude",
                longitude="Longitude",
                zoom=10,
                height=420
            )

            st.caption(
                f"Displaying up to {len(map_data):,} individual crime locations."
            )

        else:

            st.info(
                "No valid crime locations for the selected filters."
            )


    # ======================================
    # 14. GEOGRAPHICAL CRIME ANALYSIS
    # ======================================

    st.divider()

    st.markdown("## 🗺️ Geographical Crime Analysis")

    st.caption(
        "Compare crime incidents across Chicago community areas."
    )


    with st.spinner("Analyzing geographical crime distribution..."):

        community_data = con.execute(
            f"""
            SELECT
                "Community Area" AS Community_Area,
                COUNT(*) AS Incidents
            FROM crimes
            WHERE {where_clause}
              AND "Community Area" IS NOT NULL
            GROUP BY "Community Area"
            ORDER BY Incidents DESC
            LIMIT 10
            """,
            params
        ).fetchdf()


    # --------------------------------------
    # GRAPH 6: TREEMAP
    # --------------------------------------

    if not community_data.empty:

        st.markdown("### Top 10 Crime-Affected Community Areas")

        community_data["Community_Area"] = (
            community_data["Community_Area"].astype(int).astype(str)
        )

        treemap_fig = px.treemap(
            community_data,
            path=["Community_Area"],
            values="Incidents",
            color="Incidents",
            labels={
                "Community_Area": "Community Area",
                "Incidents": "Crime Incidents"
            }
        )

        treemap_fig.update_traces(
            textinfo="label+value+percent parent"
        )

        treemap_fig.update_layout(
            height=380,
            margin=dict(l=10, r=10, t=20, b=10),
            paper_bgcolor="#0B1020",
            font=dict(color="#CBD5E1")
        )

        st.plotly_chart(
            treemap_fig,
            use_container_width=True
        )

        st.caption(
            "Community area numbers represent geographical "
            "areas within Chicago, not crime district numbers."
        )

    else:

        st.info("No community area data available.")


    # ======================================
    # 15. FOOTER
    # ======================================

    st.divider()

    st.caption(
        "Night Watch | Crime Intelligence & Hotspot Mapping | "
        "Chicago Crime Incidents Dataset"
    )


except Exception as e:

    st.error(f"Dashboard error: {e}")

Overwriting app.py


In [111]:
import shutil

shutil.copy(
    "app.py",
    "/content/drive/MyDrive/app.py"
)

print("Python code saved successfully!")

Python code saved successfully!


In [109]:

import os
import subprocess
import time
import requests

# Stop the old Streamlit server
subprocess.run(
    ["bash", "-lc", "fuser -k 8501/tcp 2>/dev/null || true"]
)

# Preserve dataset location
os.environ["CRIME_CSV"] = DATA_PATH

# Start Streamlit with Colab-compatible settings
log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "python", "-m", "streamlit", "run", "app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
        "--browser.gatherUsageStats=false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(8)

print("Streamlit server started!")

Streamlit server started!


In [110]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=900)

<IPython.core.display.Javascript object>